<a href="https://colab.research.google.com/github/yashicam22/Stock-Scorecard/blob/main/Week4_Stock_Technical_Indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose stocks_13_labeled.csv

In [ ]:
# First compute daily return
df['Return'] = df.groupby('Ticker')['Close'].pct_change()

# Moving averages
df['SMA20'] = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(20).mean())
df['SMA50'] = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(50).mean())
df['EMA20'] = df.groupby('Ticker')['Close'].transform(lambda s: s.ewm(span=20, adjust=False).mean())

# VWAP20 (20-day rolling volume-weighted average price)
df['TP'] = (df['High'] + df['Low'] + df['Close']) / 3  # typical price
df['TPV'] = df['TP'] * df['Volume']
df['VWAP20'] = (
    df.groupby('Ticker')['TPV'].transform(lambda s: s.rolling(20).sum()) /
    df.groupby('Ticker')['Volume'].transform(lambda s: s.rolling(20).sum())
)

# MACD (12-26 EMA) and Signal (9-EMA of MACD)
def compute_macd(close):
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return macd, signal

macd_list, signal_list = [], []
for ticker, g in df.groupby('Ticker'):
    macd, signal = compute_macd(g['Close'])
    macd_list.append(macd)
    signal_list.append(signal)
df['MACD'] = pd.concat(macd_list)
df['Signal'] = pd.concat(signal_list)

# RSI14
def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

rsi_list = []
for ticker, g in df.groupby('Ticker'):
    rsi_list.append(compute_rsi(g['Close']))
df['RSI14'] = pd.concat(rsi_list)

df = df.drop(columns=['TP', 'TPV'])  # cleanup helper columns

print(df[['SMA20','SMA50','EMA20','VWAP20','MACD','Signal','RSI14']].describe())

In [ ]:
# Reset to a unique row index so per-ticker results can be reassigned safely
df = df.reset_index()  # 'Date' becomes a normal column; index is now 0,1,2,...unique

# MACD (12-26 EMA) and Signal (9-EMA of MACD)
def compute_macd(close):
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return macd, signal

macd_list, signal_list = [], []
for ticker, g in df.groupby('Ticker'):
    macd, signal = compute_macd(g['Close'])
    macd_list.append(macd)
    signal_list.append(signal)
df['MACD'] = pd.concat(macd_list)
df['Signal'] = pd.concat(signal_list)

# RSI14
def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

rsi_list = []
for ticker, g in df.groupby('Ticker'):
    rsi_list.append(compute_rsi(g['Close']))
df['RSI14'] = pd.concat(rsi_list)

# Put Date back as the index
df = df.set_index('Date').sort_index()

df = df.drop(columns=['TP', 'TPV'], errors='ignore')  # cleanup helper columns if present

print(df[['SMA20','SMA50','EMA20','VWAP20','MACD','Signal','RSI14']].describe())

In [ ]:
df['ret_5d']   = df.groupby('Ticker')['Close'].pct_change(5)
df['ret_20d']  = df.groupby('Ticker')['Close'].pct_change(20)
df['px_sma20'] = df['Close'] / df['SMA20'] - 1
df['px_sma50'] = df['Close'] / df['SMA50'] - 1
df['ema_sma']  = df['EMA20'] / df['SMA20'] - 1
df['px_vwap']  = df['Close'] / df['VWAP20'] - 1
df['macd_n']   = (df['MACD'] - df['Signal']) / df['Close']
df['vol_20']   = df.groupby('Ticker')['Return'].transform(lambda s: s.rolling(20).std())
df['rel_vol']  = df['Volume'] / df.groupby('Ticker')['Volume'].transform(lambda s: s.rolling(20).mean())

FEATURES = ['ret_5d','ret_20d','px_sma20','px_sma50','ema_sma',
            'px_vwap','macd_n','vol_20','rel_vol','RSI14']

df[FEATURES].describe()

In [ ]:
model_df = df.dropna(subset=FEATURES + ['Label'])
print(f"Rows before: {len(df)}, after dropna: {len(model_df)}")

In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import precision_score

train = model_df[model_df.index < '2024-01-01']
test  = model_df[model_df.index >= '2024-01-01']

print(f"Train rows: {len(train)}, Test rows: {len(test)}")

In [ ]:
print("Data starts:", model_df.index.min())
print("Data ends:", model_df.index.max())

In [ ]:
model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05)
model.fit(train[FEATURES], train['Label'])

pred = model.predict(test[FEATURES])

win_rate = test['Label'].mean()
model_precision = precision_score(test['Label'], pred)

print('Win rate in test years :', round(win_rate, 3))
print('Model precision        :', round(model_precision, 3))

In [ ]:
mom_pred = (test['ret_20d'] > 0).astype(int)
mom_precision = precision_score(test['Label'], mom_pred)
print('Momentum precision:', round(mom_precision, 3))

In [ ]:
import pandas as pd

In [ ]:
comparison = pd.DataFrame({
    'Actual': test['Label'].values,
    'Predicted': pred
}, index=test.index)

comparison.head(20)

In [ ]:
%who

np	 pd	 
